In [2]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    # .config("spark.driver.memory", "4g")
    .appName("Reading and Parsing JSON Files/Data")
    .master("local[*]")
    .getOrCreate()
)

spark


In [8]:
df_single = spark.read.format("json").load("scratch/datasets/order_singleline.json")

In [9]:
df_single.printSchema() 

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)



In [10]:
df_single.show(truncate=False)

+------------------------+-----------+--------+------------------------------------+
|contact                 |customer_id|order_id|order_line_items                    |
+------------------------+-----------+--------+------------------------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|
+------------------------+-----------+--------+------------------------------------+



In [12]:
df_multi = spark.read.format("json").option("multiline", "true").load("scratch/datasets/order_multiline.json")

In [13]:
df_multi.printSchema()

root
 |-- contact: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)



In [14]:
df_multi.show(truncate=False)

+------------------------+-----------+--------+------------------------------------+
|contact                 |customer_id|order_id|order_line_items                    |
+------------------------+-----------+--------+------------------------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|
+------------------------+-----------+--------+------------------------------------+



In [3]:
df = spark.read.format("text").load("scratch/datasets/order_singleline.json")

In [5]:
df.printSchema()

root
 |-- value: string (nullable = true)



In [ ]:
_schema = "customer_id string, order_id string, cntact array<long>"
df_schema = spark.read.format("json").schema(_schema).load("scratch/datasets/order_singleline.json")

In [ ]:
df_schema.show(truncate=False)

+-----------+--------+------------------------+
|customer_id|order_id|contact                 |
+-----------+--------+------------------------+
|C001       |O101    |[9000010000, 9000010001]|
+-----------+--------+------------------------+



In [20]:
_schema = "contact array<string>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>"

In [22]:
df_schema_new = spark.read.format("json").schema(_schema).load("scratch/datasets/order_singleline.json")

In [24]:
df_schema_new.printSchema()

root
 |-- contact: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- qty: long (nullable = true)



In [23]:
df_schema_new.show(truncate=False)

+------------------------+-----------+--------+------------------------------------+
|contact                 |customer_id|order_id|order_line_items                    |
+------------------------+-----------+--------+------------------------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|
+------------------------+-----------+--------+------------------------------------+



In [6]:
_schema = "contact array<string>, customer_id string, order_id string, order_line_items array<struct<amount double, item_id string, qty long>>"
from pyspark.sql.functions import from_json, col

df_expanded = df.withColumn("parsed", from_json(col("value"), _schema))

In [7]:
df_expanded.printSchema()

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)



In [8]:
df_expanded.show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------+
|value                                                                                                                                                                              |parsed                                                                      |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------+
|{"order_id":"O101","customer_id":"C001","order_line_items":[{"item_id":"I001","qty":6,"amount":102.45},{"item_id":"I003","qty":2,"amount":2.01}],"contact":[9000010000,9000010001]}|{[9000010000, 9000010001], C001, O101, [{1

In [10]:
# Function to_json to parse a JSON string
from pyspark.sql.functions import to_json

df_unparsed = df_expanded.withColumn("unparsed", to_json(df_expanded.parsed))

In [11]:
df_unparsed.printSchema()

root
 |-- value: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- contact: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- order_id: string (nullable = true)
 |    |-- order_line_items: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- amount: double (nullable = true)
 |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |-- qty: long (nullable = true)
 |-- unparsed: string (nullable = true)



In [12]:
df_unparsed.select("unparsed").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|unparsed                                                                                                                                                                               |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"contact":["9000010000","9000010001"],"customer_id":"C001","order_id":"O101","order_line_items":[{"amount":102.45,"item_id":"I001","qty":6},{"amount":2.01,"item_id":"I003","qty":2}]}|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [13]:
df_1 = df_expanded.select("parsed.*")

In [15]:
df_1.show(truncate=False)

+------------------------+-----------+--------+------------------------------------+
|contact                 |customer_id|order_id|order_line_items                    |
+------------------------+-----------+--------+------------------------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|
+------------------------+-----------+--------+------------------------------------+



In [16]:
from pyspark.sql.functions import explode
df_2 = df_1.withColumn("expanded_line_items", explode("order_line_items"))

In [17]:
df_2.show(truncate=False)

+------------------------+-----------+--------+------------------------------------+-------------------+
|contact                 |customer_id|order_id|order_line_items                    |expanded_line_items|
+------------------------+-----------+--------+------------------------------------+-------------------+
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|{102.45, I001, 6}  |
|[9000010000, 9000010001]|C001       |O101    |[{102.45, I001, 6}, {2.01, I003, 2}]|{2.01, I003, 2}    |
+------------------------+-----------+--------+------------------------------------+-------------------+



In [21]:
df_3 = df_2.select("contact", "customer_id", "order_id", "order_id", "expanded_line_items.*")

In [22]:
df_3.show(truncate=False)

+------------------------+-----------+--------+--------+------+-------+---+
|contact                 |customer_id|order_id|order_id|amount|item_id|qty|
+------------------------+-----------+--------+--------+------+-------+---+
|[9000010000, 9000010001]|C001       |O101    |O101    |102.45|I001   |6  |
|[9000010000, 9000010001]|C001       |O101    |O101    |2.01  |I003   |2  |
+------------------------+-----------+--------+--------+------+-------+---+



In [23]:
df_final = df_3.withColumn("contact_expanded", explode("contact"))

In [24]:
df_final.show(truncate=False)

+------------------------+-----------+--------+--------+------+-------+---+----------------+
|contact                 |customer_id|order_id|order_id|amount|item_id|qty|contact_expanded|
+------------------------+-----------+--------+--------+------+-------+---+----------------+
|[9000010000, 9000010001]|C001       |O101    |O101    |102.45|I001   |6  |9000010000      |
|[9000010000, 9000010001]|C001       |O101    |O101    |102.45|I001   |6  |9000010001      |
|[9000010000, 9000010001]|C001       |O101    |O101    |2.01  |I003   |2  |9000010000      |
|[9000010000, 9000010001]|C001       |O101    |O101    |2.01  |I003   |2  |9000010001      |
+------------------------+-----------+--------+--------+------+-------+---+----------------+



In [25]:
df_final.printSchema()

root
 |-- contact: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- item_id: string (nullable = true)
 |-- qty: long (nullable = true)
 |-- contact_expanded: string (nullable = true)



In [26]:
df_final.drop("contact").show(truncate=False)

+-----------+--------+--------+------+-------+---+----------------+
|customer_id|order_id|order_id|amount|item_id|qty|contact_expanded|
+-----------+--------+--------+------+-------+---+----------------+
|C001       |O101    |O101    |102.45|I001   |6  |9000010000      |
|C001       |O101    |O101    |102.45|I001   |6  |9000010001      |
|C001       |O101    |O101    |2.01  |I003   |2  |9000010000      |
|C001       |O101    |O101    |2.01  |I003   |2  |9000010001      |
+-----------+--------+--------+------+-------+---+----------------+



In [ ]:
# BONUS TIP

# Identify schema from the JSON string

# schema_of_json